In [2]:
import pandas as pd

# Reading the first hospital invoice
invoices_h1 = pd.read_csv('../data/invoices/hospital_1_invoices.csv')
line_items_h1 = pd.read_csv('../data/invoices/hospital_1_line_items.csv')
labels_h1 = pd.read_csv('../data/labels/hospital_1_labels.csv')

# Integrating the invoice with its details to understand the full picture.
merged_data = line_items_h1.merge(invoices_h1, on='invoice_id', how='left')

# show first 5 row
merged_data.head()

,line_id,invoice_id,line_no,service_date,description,quantity,unit_basis_as_billed,unit_price_cents,line_total_cents,hospital_id,contract_number,invoice_date,patient_id,facility_code,plan_tier,admission_date,discharge_date,invoice_total_cents
0,H1-L00001-01,INV-H1-000001,1,2024-01-01,Procedure Routine Urologic Biop /NG-3022,1,per_procedure,226950,226950,H1,INS-H1-2024-0417,2024-01-05,PT-H1-000013,F-MAIN,BRONZE,2024-01-01,2024-01-02,2475525
1,H1-L00001-02,INV-H1-000001,2,2024-01-01,Case - Intensive Ophth /NG-3641,13,per_hour,21875,284375,H1,INS-H1-2024-0417,2024-01-05,PT-H1-000013,F-MAIN,BRONZE,2024-01-01,2024-01-02,2475525
2,H1-L00001-03,INV-H1-000001,3,2024-01-01,Visit Amb Hm,1,per_visit,8225,8225,H1,INS-H1-2024-0417,2024-01-05,PT-H1-000013,F-MAIN,BRONZE,2024-01-01,2024-01-02,2475525
3,H1-L00001-04,INV-H1-000001,4,2024-01-01,Elective Pulmonary Nutr Support,7,per_unit_dispensed,7450,52150,H1,INS-H1-2024-0417,2024-01-05,PT-H1-000013,F-MAIN,BRONZE,2024-01-01,2024-01-02,2475525
4,H1-L00001-05,INV-H1-000001,5,2024-01-01,Fraction Std Otolaryngologic Radiother,11,per_item,25250,277750,H1,INS-H1-2024-0417,2024-01-05,PT-H1-000013,F-MAIN,BRONZE,2024-01-01,2024-01-02,2475525


In [5]:
import os

# Define the correct path to the contract file based on the actual file name
contract_path = '../data/contracts/hospital_1/provider_services_agreement.md' 

# Open and read the contract file
with open(contract_path, 'r', encoding='utf-8') as file:
    hospital_1_contract = file.read()

# Print the first 500 characters to inspect the contract's structure
print(hospital_1_contract[:500])

# Provider Services Agreement

_Schedule of Contracted Services and Rates_

**Contract number:** INS-H1-2024-0417
**Provider:** Northgate Regional Medical Centre
**Payer:** Meridian Health Assurance Group
**Effective from:** 1 January 2024
**Effective to:** 31 December 2025
**Currency:** GBP
**Rounding convention:** half_up_cent

## 1. Parties and Term

This Agreement is made between Meridian Health Assurance Group (the "Payer") and Northgate Regional Medical Centre (the "Provider").

1.1 The Ag


In [12]:
import os
import json
import google.generativeai as genai
from dotenv import load_dotenv

# 1. Load environment variables (your Gemini API Key)
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
genai.configure(api_key=api_key)

# 2. Initialize the Gemini model 
model = genai.GenerativeModel('gemini-3.1-flash-lite')

# 3. Define the Prompt 
prompt = """
You are an expert medical billing auditor. I will provide you with a hospital provider services agreement.
Your task is to extract the pricing schedule and billing rules into a structured JSON format.

Extract the following details:
1. contract_metadata: provider name, contract number, effective dates, currency.
2. services: a list of services where each service has:
   - service_name: The official name of the service in the contract.
   - unit_basis: The basis for billing (e.g., per_procedure, per_hour, per_visit, per_diem).
   - unit_price_cents: The price in cents (integer). If the price in the contract is in major currency (e.g., 150.00), convert it to cents (15000).
   - conditions: Any special conditions, limits, or discounts mentioned for this specific service (leave empty string if none).

Return ONLY a valid JSON object. Do not include markdown blocks like ```json ... ```.

Here is the contract:
"""

# 4. Save the prompt to a file (Required by the assignment to show your work)
os.makedirs('prompts', exist_ok=True)
with open('prompts/prompt_v1.txt', 'w', encoding='utf-8') as f:
    f.write(prompt)

print("⏳ Sending contract to Gemini API to extract rules... Please wait.")

# 5. Call the API by combining the prompt and the contract text
response = model.generate_content(prompt + "\n\n" + hospital_1_contract)

# 6. Clean the response to ensure it's pure JSON
extracted_text = response.text.strip()
if extracted_text.startswith("```json"):
    extracted_text = extracted_text[7:-3].strip()
elif extracted_text.startswith("```"):
    extracted_text = extracted_text[3:-3].strip()

# 7. Parse the JSON and save it to a file
try:
    contract_rules = json.loads(extracted_text)
    
    # Save the structured data
    output_path = '../data/contracts/hospital_1/contract_rules.json'
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(contract_rules, f, indent=4)
        
    print(f"✅ Contract rules successfully extracted and saved to {output_path}")
    print("\nPreview of the extracted services:")
    
    # Print a quick preview of the first 2 services extracted
    for service in contract_rules.get("services", [])[:2]:
        print(f"- {service['service_name']}: {service['unit_price_cents']} cents ({service['unit_basis']})")

except json.JSONDecodeError:
    print("❌ Failed to parse JSON. The model might have returned text instead of strict JSON.")
    print("Raw output:")
    print(extracted_text)

⏳ Sending contract to Gemini API to extract rules... Please wait.
✅ Contract rules successfully extracted and saved to ../data/contracts/hospital_1/contract_rules.json

Preview of the extracted services:
- Advanced Cardiac Recovery Room Occupancy: 20000 cents (per_hour)
- Advanced Haematology Physiotherapy Session: 10150 cents (per_hour)


In [8]:
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/lyria-3.5
models/gemini-3.1-flash-tts-preview
models/

In [14]:
"""
Script Function: Item Mapping (Fuzzy Matching)
This script matches the raw billing descriptions from invoices to the official contract terms.
It saves the output to 'service_mapping.json' so the auditor engine can use it.
"""

import pandas as pd
import json
from rapidfuzz import process, fuzz

# 1. Setup relative paths
line_items_path = '../data/invoices/hospital_1_line_items.csv'
contract_rules_path = '../data/contracts/hospital_1/contract_rules.json'
mapping_output_path = '../data/contracts/hospital_1/service_mapping.json'

# 2. Load data
line_items_df = pd.read_csv(line_items_path)
with open(contract_rules_path, 'r', encoding='utf-8') as f:
    contract_rules = json.load(f)

# 3. Extract names
official_services = [service['service_name'] for service in contract_rules['services']]
billed_descriptions = line_items_df['description'].unique()

print("⏳ Mapping billed descriptions to official contract services...")

# 4. Perform Fuzzy Matching
mapping_dict = {}

for billed_desc in billed_descriptions:
    # Clean description before matching
    clean_desc = billed_desc.split('/')[0].strip() if '/' in billed_desc else billed_desc
    
    # Find the best match
    best_match = process.extractOne(clean_desc, official_services, scorer=fuzz.token_sort_ratio)
    
    if best_match:
        matched_service, score, _ = best_match
        mapping_dict[billed_desc] = {
            "mapped_service": matched_service,
            "confidence_score": round(score, 2)
        }

# 5. Save the mapping to a JSON file
with open(mapping_output_path, 'w', encoding='utf-8') as f:
    json.dump(mapping_dict, f, indent=4)

print(f"✅ Mapping complete! File saved successfully at: {mapping_output_path}")

⏳ Mapping billed descriptions to official contract services...
✅ Mapping complete! File saved successfully at: ../data/contracts/hospital_1/service_mapping.json


In [ ]:
"""
Script Function: Invoice Auditor Engine
This script applies the contract rules (including Thresholds and Caps) to the billed line items.
It calculates the 'expected' price for each line and aggregates it to the invoice level.
Finally, it compares the expected total with the billed total to flag erroneous invoices.
"""

import pandas as pd
import json
import re

# 1. Setup relative paths
invoices_path = '../data/invoices/hospital_1_invoices.csv'
line_items_path = '../data/invoices/hospital_1_line_items.csv'
contract_rules_path = '../data/contracts/hospital_1/contract_rules.json'
mapping_path = '../data/contracts/hospital_1/service_mapping.json'

# 2. Load data
invoices_df = pd.read_csv(invoices_path)
line_items_df = pd.read_csv(line_items_path)

with open(contract_rules_path, 'r', encoding='utf-8') as f:
    contract_rules = json.load(f)

with open(mapping_path, 'r', encoding='utf-8') as f:
    service_mapping = json.load(f)

# Convert rules list to a dictionary for fast lookup by service name
rules_dict = {item['service_name']: item for item in contract_rules['services']}

# 3. Define the pricing logic function (Handling Thresholds and Caps)
def calculate_expected_line_total(quantity, base_price, condition_str):
    """
    Calculates the expected total for a line item based on its quantity, base price, and contract conditions.
    """
    # If there are no special conditions, it's just a simple multiplication
    if not condition_str or pd.isna(condition_str):
        return quantity * base_price
    
    # Logic 1: Handle Threshold Premium (e.g., "Threshold premium: +20% if daily qty > 6")
    threshold_match = re.search(r'Threshold premium:\s*\+(\d+)%\s*if daily qty >\s*(\d+)', str(condition_str))
    if threshold_match:
        premium_percent = int(threshold_match.group(1)) # e.g., 20
        threshold_qty = int(threshold_match.group(2))   # e.g., 6
        
        if quantity > threshold_qty:
            base_units = threshold_qty
            premium_units = quantity - threshold_qty
            # Calculate premium price (integer math for cents)
            premium_price = round(base_price * (1 + (premium_percent / 100.0)))
            return (base_units * base_price) + (premium_units * premium_price)
        else:
            return quantity * base_price

    # Logic 2: Handle Daily Cap (e.g., "Daily cap: 6 days" or "Daily cap: 4 tests")
    cap_match = re.search(r'Daily cap:\s*(\d+)', str(condition_str))
    if cap_match:
        cap_qty = int(cap_match.group(1))
        # We only pay up to the cap_qty, anything above is free (0 cents)
        allowed_qty = min(quantity, cap_qty)
        return allowed_qty * base_price

    # For any other complex conditions not yet programmed, default to normal calculation
    return quantity * base_price

# 4. Apply the logic to all line items
print("⏳ Auditing line items...")
expected_line_totals = []
error_categories = []
confidences = []

for idx, row in line_items_df.iterrows():
    billed_desc = row['description']
    quantity = row['quantity']
    
    # Find the mapped official service
    mapping_data = service_mapping.get(billed_desc, {})
    official_service = mapping_data.get('mapped_service')
    confidence = mapping_data.get('confidence_score', 0)
    
    # If mapping exists, apply pricing logic
    if official_service and official_service in rules_dict and confidence >= 60:
        base_price = rules_dict[official_service]['unit_price_cents']
        condition_str = rules_dict[official_service]['conditions']
        
        expected_total = calculate_expected_line_total(quantity, base_price, condition_str)
        error_cat = "Clean"
    else:
        # If mapping failed, we flag it as unknown
        expected_total = 0
        error_cat = "Unmapped Service"
        confidence = 0.0
        
    expected_line_totals.append(expected_total)
    error_categories.append(error_cat)
    # Convert confidence from 0-100 to 0-1 scale as required by submission
    confidences.append(confidence / 100.0)

line_items_df['expected_line_total'] = expected_line_totals
line_items_df['line_error_category'] = error_categories
line_items_df['line_confidence'] = confidences

# 5. Aggregate line items up to the Invoice level
print("⏳ Aggregating to invoice level...")
invoice_summary = line_items_df.groupby('invoice_id').agg(
    expected_total_cents=('expected_line_total', 'sum'),
    min_confidence=('line_confidence', 'min') # The invoice is only as confident as its weakest mapped line
).reset_index()

# Merge with the main invoices dataframe
final_audit_df = invoices_df.merge(invoice_summary, on='invoice_id', how='left')

# 6. Flag the errors
final_audit_df['expected_total_cents'] = final_audit_df['expected_total_cents'].fillna(0)
final_audit_df['min_confidence'] = final_audit_df['min_confidence'].fillna(0.0)
final_audit_df['flagged'] = (abs(final_audit_df['invoice_total_cents'] - final_audit_df['expected_total_cents']) > 2).astype(int)
# Simple categorization for the flagged invoices
def categorize_error(row):
    if row['flagged'] == 0:
        return ""
    if row['min_confidence'] < 0.6:
        return "Low Confidence Mapping"
    if row['invoice_total_cents'] > row['expected_total_cents']:
        return "Overbilled (Rate/Threshold error)"
    return "Underbilled"

final_audit_df['error_category'] = final_audit_df.apply(categorize_error, axis=1)

print("✅ Audit complete! Here is a preview of the flagged invoices:")
flagged_preview = final_audit_df[final_audit_df['flagged'] == 1][['invoice_id', 'invoice_total_cents', 'expected_total_cents', 'error_category', 'min_confidence']].head()
print(flagged_preview)

⏳ Auditing line items...
⏳ Aggregating to invoice level...
✅ Audit complete! Here is a preview of the flagged invoices:
      invoice_id  invoice_total_cents  expected_total_cents  \
0  INV-H1-000001              2475525               2495725   
1  INV-H1-000002              2588836               2580400   
2  INV-H1-000003              5777580               4808150   
3  INV-H1-000004              1260398                924200   
5  INV-H1-000006              1746150               1741750   

           error_category  min_confidence  
0  Low Confidence Mapping          0.5161  
1  Low Confidence Mapping          0.4643  
2  Low Confidence Mapping          0.4490  
3  Low Confidence Mapping          0.5484  
5  Low Confidence Mapping          0.4643  


In [19]:
"""
Script Function: Evaluate Auditor against Ground Truth (Hospital 1 only)
This script compares our predicted 'flagged' column against the actual 'is_erroneous' column.
"""
import pandas as pd

# 1. Load the ground truth labels
labels_path = '../data/labels/hospital_1_labels.csv'
labels_df = pd.read_csv(labels_path)

# 2. Merge our final_audit_df with the labels
evaluation_df = final_audit_df[['invoice_id', 'flagged']].merge(
    labels_df[['invoice_id', 'is_erroneous']], 
    on='invoice_id'
)

# 3. Calculate Evaluation Metrics
tp = len(evaluation_df[(evaluation_df['flagged'] == 1) & (evaluation_df['is_erroneous'] == 1)])
fp = len(evaluation_df[(evaluation_df['flagged'] == 1) & (evaluation_df['is_erroneous'] == 0)])
fn = len(evaluation_df[(evaluation_df['flagged'] == 0) & (evaluation_df['is_erroneous'] == 1)])
tn = len(evaluation_df[(evaluation_df['flagged'] == 0) & (evaluation_df['is_erroneous'] == 0)])

print("📊 Evaluation Results for Hospital 1:")
print(f"True Positives (Correctly flagged errors): {tp}")
print(f"False Positives (Wrongly flagged as errors): {fp}")
print(f"False Negatives (Missed errors): {fn}")
print(f"True Negatives (Correctly identified as clean): {tn}")

# Calculate Precision and Recall
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0

print("-" * 40)
print(f"Precision: {precision:.2f} (When we flag an error, how often are we right?)")
print(f"Recall: {recall:.2f} (Out of all real errors, how many did we catch?)")

📊 Evaluation Results for Hospital 1:
True Positives (Correctly flagged errors): 63
False Positives (Wrongly flagged as errors): 788
False Negatives (Missed errors): 0
True Negatives (Correctly identified as clean): 67
----------------------------------------
Precision: 0.07 (When we flag an error, how often are we right?)
Recall: 1.00 (Out of all real errors, how many did we catch?)
